In [1]:
def find_norm_soln(D, target):
    # finding all solutions to norm=target in maximal order of Q(sqrt(D))
    # Pari already shows the solutions only up to units: 
    # https://pari.math.u-bordeaux.fr/dochtml/html/Arithmetic_functions.html#qfbsolve
    # We expect D to be negative and squarefree
    # target is l*p in our case
    # see also https://ask.sagemath.org/question/76116/finding-all-integer-solutions-of-binary-quadratic-form/
    if D % 4 == 1:
        return pari.qfbsolve(pari.Qfb(1,1,(1-D)/4), target, 3).sage()
    else:
        return pari.qfbsolve(pari.Qfb(1,0,-D), target, 3).sage()


def cl_number_descent(cl, c, D):
    # cl is class number of maximal order
    # c is conductor
    # D is like above of maximal order
    unit_index = 1
    if D == -1 and c != 1:
        unit_index = 2
    elif D == -3 and c != 1:
        unit_index = 3
    
    product = 1
    primes = list(c.factor())
    for i in primes:
        if i[0] == 2:
            legendre = 0
            if D % 8 == 1:
                legendre = 1
            elif D % 8 == 5:
                legendre = -1
            product = product * (2 - legendre)/2
        else:
            product = product * (i[0] - kronecker(D,i[0]))/i[0]
    
    return cl*c*product/unit_index

In [2]:
def orders_in_single_field(D, target):
    # Find all orders in the imaginary quadratic field Q(sqrt(D)) that have an element of norm target (i.e. l*p)
    # Returns a dictionary where the keys are the conductors of these orders 
    # and the value is the number of distinct principal ideals of norm target in this order
    # The conductors can never be divisible by either p or l so we automatically have that the primes are invertible
    solns = find_norm_soln(D,target)
    
    more_solns = []
    # only needed in the cases D=-1 and D=-3
    
    conductor_list = []
    
    if len(solns) == 0:
        return None
    elif D == -1:#worry about less units when going down
        for s in solns:
            more_solns.append(s)
            if not [-1*s[1],s[0]] in solns:#PARI gives solutions up to units in Z[i] so this should always be true
                more_solns.append([-1*s[1],s[0]])
        for soln in more_solns:
            b = soln[1]
            divs_except_1 = divisors(b)
            divs_except_1.remove(1)
            conductor_list = conductor_list + divs_except_1
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
        conductor_w_mults.update({1: len(solns)})
    elif D == -3:#same as the D=-1 case
        for s in solns:
            more_solns.append(s)
            if not [-1*s[1], s[0] + s[1]] in solns:
                more_solns.append([-1*s[1], s[0] + s[1]])
            if not [-1*(s[0] + s[1]), s[0]] in solns:
                more_solns.append([-1*(s[0] + s[1]), s[0]])
        for soln in more_solns:
            b = soln[1]
            divs_except_1 = divisors(b)
            divs_except_1.remove(1)
            conductor_list = conductor_list + divs_except_1
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
        conductor_w_mults.update({1: len(solns)})
    else:
        for soln in solns:
            b = soln[1]
            conductor_list = conductor_list + divisors(b)
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
    # Make a dictionary with as keys the different conductors that appear and as values their multiplicity
    return conductor_w_mults

In [3]:
def all_norm_solutions(l,p):
    all_orders = []
    for D in list(range(-l*p,0)):
        if D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l*p)
        if order_dict != None:
            all_orders.append((D, order_dict))
    for D in list(range(-4*l*p,0)):
        if not D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l*p)
        if order_dict != None:
            all_orders.append((D, order_dict))
    return all_orders

In [4]:
def hilbert_poly(D,c):
    # Calculate the Hilbert class polynomial of the order in Q(sqrt(D)) with conductor c
    if D % 4 != 1:
        d = 4*D*c^2
    else:
        d = D*c^2
    return hilbert_class_polynomial(d)

In [5]:
def new_fix_pts(orders, l, p):
    # orders is list of tuples like all_norm_solutions outputs
    # Breaks symmetry between l and p
    # Outputs a datatype like orders, but the multiplicities are now the actual geometric ones
    for o in orders:
        leg = 1 #Legendre symbol (o[0]|p) (at this point we are assuming that p>2)
        if o[0] % p == 0:
            leg = 0
        for c in o[1].keys():
            if o[1][c] == 1:
                o[1].update({c: 2 - leg})
            elif o[1][c] == 2:
                o[1].update({c: 2*(2- leg )})
    return orders

In [6]:
def genus_X_0(p):
    #ASSUMES THAT p IS PRIME
    k = p % 3
    l = p % 4
    v_3 = 0
    v_2 = 0
    if k == 1:
        v_3 = 2
    elif k == 0:
        v_3 = 1
    if l == 1:
        v_2 = 2
    elif l == 2:
        v_2 = 1
    return int((p+1 - 3*v_2 - 4*v_3)/12)

In [7]:
def genus_fricke_quot(p):
    #ASSUMES THAT p IS PRIME
    g = genus_X_0(p)
    K = QuadraticField(-p, 'a')
    cl = K.class_number()
    fix_pt_deg = cl
    if p == 2 or p == 3:
        return 0
    if p % 4 == 3:
        unit_index = 1
        is2square = 0
        if p == 3:
            unit_index = 1/3
        if p % 8 == 7:
            is2square = 1
        if p % 8 == 3:
            is2square = -1
        #class number of order with conductor 2
        cl2 = cl * 2 * unit_index * (1- is2square/2)
        fix_pt_deg = cl + cl2
    return int((g + 1 - fix_pt_deg/2)/2)

In [8]:
def shadow_poly(p):
    # Calculate the polynomial associated to a certain multiple n of the shadow of T_2 on X_0^+(p)
    # Which multiple depends on the class of p mod 12
    # Assume p > 7

    
    p12 = p % 12
    p8 = p % 8
    p7 = p % 7
    p4 = p % 4

    if p12 == 11:
        n = 2 
    if p12 == 7:
        n = 6
    if p12 == 5:
        n = 4        
    if p12 == 1:
        n = 12 

    gplus = genus_fricke_quot(p)

    # We will return a list of tuples, containing a polynomial mod p and an integer to which we should raise that polynomial
    list_of_factors = []

    # We consider the fixed points which were already fixed points on X_0(p)
    R.<x> = PolynomialRing(ZZ)

    # (in the mean time we keep track of the degree of the total fixed point divisor)
    # (we start with 4 since this is the degree we get from the cusp)
    deg_fix_pts = 4

    if p7 in [1,2,4]:
        list_of_factors.append(((x + 3375).change_ring(GF(p)), n*(2*gplus - 2)*4))
        deg_fix_pts += 4
        #DOUBLE-CHECK THESE DEGREES
    if p8 in [1,3]:
        list_of_factors.append(((x - 8000).change_ring(GF(p)), n*(2*gplus - 2)*2))
        deg_fix_pts += 2
    if p4 == 1:
        list_of_factors.append(((x- 1728).change_ring(GF(p)), n*(2*gplus - 2)*2))
        deg_fix_pts += 2
    #print(list_of_factors)


    # Next the new fixed points
    new_fix_points = new_fix_pts(all_norm_solutions(2,p), 2, p)

    for o in new_fix_points:
        if o[0] % p != 0:# This is equivalent to p being unramified (since p > 2)
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]))
                deg_fix_pts += o[1][c]
        else:
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]/2))
                deg_fix_pts += o[1][c]/2

    # Coefficient of the term 'n K_X' in the shadow
    deg_nK = 12 - deg_fix_pts
    

    # The fixed points from the Fricke involution 
    # (Isn't is faster to compute supersingular j's directly using this SupersingularModule?)
    fricke_fix_pts = []
    fricke_fix_pts.append(hilbert_poly(-p, 1).change_ring(GF(p)))
    if p4 == 3:
        fricke_fix_pts.append(hilbert_poly(-p, 2).change_ring(GF(p)))

    # All supersingular j-invariants
    ssj = SupersingularModule(p).supersingular_points()[0]
    # Only the F_p rational supersingular j-invariants
    rat_ssj = []
    K = parent(ssj[0])
    phi = K.frobenius_endomorphism(1)
    for j in ssj:
        if j == phi(j):
            rat_ssj.append(j)

    S.<y> = PolynomialRing(GF(p))
    
    T.<X,Y> = GF(p)[]
    mod_poly2 = X^3 - X^2 * Y^2 + 1488*X^2 * Y - 162000*X^2 + 1488*X*Y^2 + 40773375*X*Y + 8748000000*X + Y^3 - 162000*Y^2 + 8748000000*Y - 157464000000000
    
    if p12 == 11:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -1* deg_nK))
        # The '- 2 T_2(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly2(X,j).univariate_polynomial(), 8 ))

    if p12 == 7:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -3* deg_nK))
        list_of_factors.append((x.change_ring(GF(p)), -4* deg_nK))
        # The '- 2 T_2(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly2(X,j).univariate_polynomial(), 24))
        list_of_factors.append((mod_poly2(X,0).univariate_polynomial(), 16))

    if p12 == 5:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -2* deg_nK))
        list_of_factors.append(((x-1728).change_ring(GF(p)), -2* deg_nK))
        # The '- 2 T_2(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly2(X,j).univariate_polynomial(), 16))
        list_of_factors.append((mod_poly2(X,1728).univariate_polynomial(), 8))
        
    if p12 == 1:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -6* deg_nK))
        list_of_factors.append(((x-1728).change_ring(GF(p)), -6* deg_nK))
        list_of_factors.append((x.change_ring(GF(p)), -8* deg_nK))
        # The '- 2 T_2(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly2(X,j).univariate_polynomial(), 48))
        list_of_factors.append((mod_poly2(X,1728).univariate_polynomial(), 24))
        list_of_factors.append((mod_poly2(X,0).univariate_polynomial(), 32))
    return (list_of_factors, ssj, rat_ssj, deg_fix_pts)

In [10]:
def T3shadow_poly(p):
    # Calculate the polynomial associated to a certain multiple n of the shadow of T_3 on X_0^+(p)
    # Which multiple depends on the class of p mod 12
    # Assume p > ...

    
    p12 = p % 12
    p8 = p % 8
    p11 = p % 11
    p4 = p % 4
    p3 = p % 3

    if p12 == 11:
        n = 2 
    if p12 == 7:
        n = 6
    if p12 == 5:
        n = 4        
    if p12 == 1:
        n = 12 

    gplus = genus_fricke_quot(p)

    # We will return a list of tuples, containing a polynomial mod p and an integer to which we should raise that polynomial
    list_of_factors = []

    # We consider the fixed points which were already fixed points on X_0(p)
    R.<x> = PolynomialRing(ZZ)

    # (in the mean time we keep track of the degree of the total fixed point divisor)
    deg_fix_pts = 4

    if p8 in [1,3]:# Corresponds to Z[sqrt(-2)]
        list_of_factors.append(((x - 8000).change_ring(GF(p)), n*(2*gplus - 2)*4))
        deg_fix_pts += 4
    if p11 in [1,3,4,5,9]:# Corresponds to Z[(1 + sqrt(-11))/2]
        list_of_factors.append(((x + 32768).change_ring(GF(p)), n*(2*gplus - 2)*4))
        deg_fix_pts += 4
    if p3 == 1:# Corresponds to both Z[(1 + sqrt(-3))/2] and Z[sqrt(-3)]
        list_of_factors.append(((x - 54000).change_ring(GF(p)), n*(2*gplus - 2)*2))
        list_of_factors.append(((x).change_ring(GF(p)), n*(2*gplus - 2)*2))
        deg_fix_pts += 4
        


    # Next the new fixed points
    new_fix_points = new_fix_pts(all_norm_solutions(3,p), 3, p)

    for o in new_fix_points:
        if o[0] % p != 0:
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]))
                deg_fix_pts += o[1][c]
        else:
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]/2))
                deg_fix_pts += o[1][c]/2

    # Coefficient of the term 'n K_X' in the shadow
    deg_nK = 16 - deg_fix_pts

    # The fixed points from the Fricke involution 
    # (Isn't is faster to compute supersingular j's directly using this SupersingularModule?)
    fricke_fix_pts = []
    fricke_fix_pts.append(hilbert_poly(-p, 1).change_ring(GF(p)))
    if p4 == 3:
        fricke_fix_pts.append(hilbert_poly(-p, 2).change_ring(GF(p)))

    # All supersingular j-invariants
    ssj = SupersingularModule(p).supersingular_points()[0]
    # Only the F_p rational supersingular j-invariants
    rat_ssj = []
    K = parent(ssj[0])
    phi = K.frobenius_endomorphism(1)
    for j in ssj:
        if j == phi(j):
            rat_ssj.append(j)

    S.<y> = PolynomialRing(GF(p))
    
    T.<X,Y> = GF(p)[]
    mod_poly3 = (X^4 - X^3 * Y^3 + 2232 * X^3 * Y^2 - 1069956 * X^3 * Y + 36864000 * X^3 + 2587918086 * X^2 * Y^2
    + 8900222976000 * X^2 * Y + 452984832000000 * X^2 - 770845966336000000 * X * Y + 1855425871872000000000 * X
    + Y^4 + 2232 * X^2 * Y^3 - 1069956 * X * Y^3 + 36864000 * Y^3 + 8900222976000 * X * Y^2 + 452984832000000 * Y^2
    + 1855425871872000000000 * Y)
    
    if p12 == 11:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -1* deg_nK))
        # The '- 2 T_3(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly3(X,j).univariate_polynomial(), 8 ))

    if p12 == 7:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -3* deg_nK))
        list_of_factors.append((x.change_ring(GF(p)), -4* deg_nK))
        # The '- 2 T_3(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly3(X,j).univariate_polynomial(), 24))
        list_of_factors.append((mod_poly3(X,0).univariate_polynomial(), 16))

    if p12 == 5:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -2* deg_nK))
        list_of_factors.append(((x-1728).change_ring(GF(p)), -2* deg_nK))
        # The '- 2 T_3(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly3(X,j).univariate_polynomial(), 16))
        list_of_factors.append((mod_poly3(X,1728).univariate_polynomial(), 8))
        
    if p12 == 1:
        # The 'n K_X' part of shadow (we don't care about the coefficient of the cusp)
        for f in fricke_fix_pts:
            list_of_factors.append((f, -6* deg_nK))
        list_of_factors.append(((x-1728).change_ring(GF(p)), -6* deg_nK))
        list_of_factors.append((x.change_ring(GF(p)), -8* deg_nK))
        # The '- 2 T_3(nK_X)' part of the shadow (again don't care about cusps)
        for j in rat_ssj:
            list_of_factors.append((mod_poly3(X,j).univariate_polynomial(), 48))
        list_of_factors.append((mod_poly3(X,1728).univariate_polynomial(), 24))
        list_of_factors.append((mod_poly3(X,0).univariate_polynomial(), 32))
    return (list_of_factors, ssj, rat_ssj)

In [11]:
def eval_shadow_poly(list_of_factors, ssj):
    #list_of_factors is first output of shadow_poly(p)
    #ssj is a supersingular j-invariant which is not in F_p
    
    prod = 1
    #new_list = []
    zero_neg_exp = 0
    zero_pos_exp = 0
    zero_zero_exp = 0
    polys_w_zeroes = []
    
    for f in list_of_factors:
        #print("f[0](ssj) = " + str(f[0](ssj)))
        #print(f[0](ssj))
        if f[0](ssj) != 0:
            prod = prod * f[0](ssj)^f[1]
        else:
            if f[1] > 0:
                zero_pos_exp += f[1]
                polys_w_zeroes.append(f)
            elif f[1] < 0:
                zero_neg_exp += f[1]
            else:
                zero_zero_exp += 1
    return (prod, zero_pos_exp, zero_neg_exp, zero_zero_exp, polys_w_zeroes)

In [12]:
def T2shadowcheck(p):
    poly_ssjs = shadow_poly(p)
    poly = poly_ssjs[0]

    not_rat_ssj = [a for a in poly_ssjs[1] if a not in poly_ssjs[2]]
    if len(not_rat_ssj) == 0:
        return "No supersingular j's outside prime field"
    else:
        
        K = parent(not_rat_ssj[0])
        phi = K.frobenius_endomorphism(1)
        for j in not_rat_ssj:
            result = eval_shadow_poly(poly, j)
            if result[1] == 0 and result[2] == 0 and result[3] == 0 and phi(result[0]) != result[0]:
                return ("j = " + str(j), "result = " + str(result[0]))
            
        return "No supersingular j works"

def T3shadowcheck(p, show_zeroes=False):
    poly_ssjs = T3shadow_poly(p)
    poly = poly_ssjs[0]

    not_rat_ssj = [a for a in poly_ssjs[1] if a not in poly_ssjs[2]]
    if len(not_rat_ssj) == 0:
        return "No supersingular j's outside prime field"
    else:
        zeroes = []
        K = parent(not_rat_ssj[0])
        phi = K.frobenius_endomorphism(1)
        for j in not_rat_ssj:
            result = eval_shadow_poly(poly, j)
            if result[1] == 0 and result[2] == 0 and result[3] == 0 and phi(result[0]) != result[0]:
                return ("j = " + str(j), "result = " + str(result[0]))
            if show_zeroes == True:
                zeroes.append(result)
        if show_zeroes == True:
            return "No supersingular j works", zeroes
        else:
            return "No supersingular j works"
                

In [13]:
primes_to_exclude = [2,3,5,7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89,101,103,107,131,167,191]

for p in prime_range(1,500):
    if p in primes_to_exclude:
        continue
    else:
        print(p, genus_fricke_quot(p), T2shadowcheck(p))

97 3 No supersingular j works
109 3 No supersingular j works
113 3 No supersingular j works
127 3 No supersingular j works
137 4 No supersingular j works
139 3 No supersingular j works
149 3 No supersingular j works
151 3 No supersingular j works
157 5 No supersingular j works
163 6 ('j = 43*a + 42', 'result = 151*a + 153')
173 4 No supersingular j works
179 3 No supersingular j works
181 5 No supersingular j works
193 7 ('j = 51*a + 15', 'result = 150*a + 93')
197 6 ('j = 114*a + 100', 'result = 176*a + 160')
199 4 No supersingular j works
211 6 ('j = 32*a + 55', 'result = 168*a + 31')
223 6 No supersingular j works
227 5 No supersingular j works
229 7 No supersingular j works
233 7 ('j = 95*a + 221', 'result = 6*a + 104')
239 3 No supersingular j works
241 7 ('j = 196*a + 105', 'result = 114*a + 211')
251 4 No supersingular j works
257 7 No supersingular j works
263 5 ('j = 228*a + 119', 'result = 166*a + 171')
269 6 ('j = 106*a + 200', 'result = 225*a + 230')
271 6 No supersingular 